# 🎬 Projet Fouille de Données — Système de Recommandation MovieLens
**Dataset** : MovieLens Small (~100k ratings, 610 users, 9724 films)  
**Approches** : Clustering (K-Means) + Fouille de graphes (networkx)  
**Auteurs** : _À compléter_

---
## ⚙️ Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires
!pip install python-louvain --quiet
!pip install scikit-learn pandas networkx matplotlib seaborn --quiet
print('✅ Toutes les dépendances sont installées')

---
# 📦 PHASE 0 — Préparation des données
### 0.1 Téléchargement du dataset MovieLens

In [ ]:
import os, zipfile, urllib.request

URL = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
ZIP = 'ml-latest-small.zip'
DATA_DIR = 'data/'

if not os.path.exists(DATA_DIR):
    print('⬇️  Téléchargement du dataset MovieLens...')
    urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP, 'r') as z:
        z.extractall('.')
    os.rename('ml-latest-small', DATA_DIR)
    os.remove(ZIP)
    print('✅ Dataset téléchargé et extrait dans data/')
else:
    print('✅ Dataset déjà présent')

### 0.2 Imports globaux

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Graphe
import networkx as nx
import community as community_louvain  # python-louvain

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('✅ Imports OK')

### 0.3 Chargement et fusion des fichiers CSV

In [ ]:
# Chargement
ratings = pd.read_csv('data/ratings.csv')
movies  = pd.read_csv('data/movies.csv')

print('=== ratings.csv ===')
print(f'  Lignes : {len(ratings):,} | Colonnes : {list(ratings.columns)}')
print(ratings.head(3))

print('\n=== movies.csv ===')
print(f'  Lignes : {len(movies):,} | Colonnes : {list(movies.columns)}')
print(movies.head(3))

### 0.4 Gestion des valeurs manquantes et doublons

In [ ]:
print('=== Valeurs manquantes ===')
print('ratings :', ratings.isnull().sum().to_dict())
print('movies  :', movies.isnull().sum().to_dict())

print('\n=== Doublons ===')
print('ratings :', ratings.duplicated().sum())
print('movies  :', movies.duplicated().sum())

# Suppression des doublons si présents
ratings = ratings.drop_duplicates()
movies  = movies.drop_duplicates()

# Fusion
df = ratings.merge(movies, on='movieId', how='left')
print(f'\n✅ Dataframe fusionné : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(df.head(3))

### 0.5 Analyse exploratoire (EDA)

In [ ]:
print('=== Statistiques clés ===')
print(f"  Utilisateurs uniques : {df['userId'].nunique()}")
print(f"  Films uniques        : {df['movieId'].nunique()}")
print(f"  Notes totales        : {len(df):,}")
print(f"  Note moyenne         : {df['rating'].mean():.2f}")
print(f"  Plage des notes      : {df['rating'].min()} → {df['rating'].max()}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution des notes
df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des notes')
axes[0].set_xlabel('Note')
axes[0].set_ylabel('Nombre')

# Nombre de notes par utilisateur
notes_par_user = df.groupby('userId').size()
axes[1].hist(notes_par_user, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Notes par utilisateur')
axes[1].set_xlabel('Nombre de notes')
axes[1].set_ylabel('Utilisateurs')

# Nombre de notes par film (top 20)
top_films = df.groupby('title').size().sort_values(ascending=False).head(20)
top_films.plot(kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Top 20 films les plus notés')
axes[2].set_xlabel('Nombre de notes')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()
print('✅ EDA terminée — graphique sauvegardé : eda_overview.png')

### 0.6 Construction de la matrice utilisateurs × films

In [ ]:
# Matrice pivot : lignes=utilisateurs, colonnes=films, valeurs=notes (NaN si pas noté)
user_movie_matrix = df.pivot_table(index='userId', columns='movieId', values='rating')

print(f'Matrice brute : {user_movie_matrix.shape[0]} utilisateurs × {user_movie_matrix.shape[1]} films')
print(f'Densité       : {user_movie_matrix.notna().mean().mean()*100:.2f}% (très creuse — normal)')

# Remplir les NaN par 0 pour le clustering
matrix_filled = user_movie_matrix.fillna(0)

# Réduction de dimensionnalité avec SVD (pour accélérer le clustering)
# On garde 50 composantes — bon compromis variance/performance
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_svd = svd.fit_transform(matrix_filled)

variance_expliquee = svd.explained_variance_ratio_.sum()
print(f'\nSVD : 50 composantes → {variance_expliquee*100:.1f}% de variance expliquée')

# Normalisation
scaler = StandardScaler()
matrix_scaled = scaler.fit_transform(matrix_svd)

print('✅ Matrice réduite et normalisée — prête pour le clustering')

---
# 🔵 PHASE 1 — Clustering des utilisateurs (K-Means)
### 1.1 Choix du nombre optimal de clusters (méthode Elbow + Silhouette)

In [ ]:
inertias   = []
silhouettes = []
K_range = range(2, 11)

print('⏳ Calcul des scores pour k=2 à k=10...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(matrix_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(matrix_scaled, labels, sample_size=300, random_state=42)
    silhouettes.append(sil)
    print(f'  k={k:2d} → inertie={km.inertia_:,.0f} | silhouette={sil:.4f}')

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2)
ax1.set_title('Méthode Elbow — Inertie')
ax1.set_xlabel('Nombre de clusters k')
ax1.set_ylabel('Inertie')
ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))

ax2.plot(list(K_range), silhouettes, 's-', color='coral', linewidth=2)
ax2.set_title('Score de silhouette')
ax2.set_xlabel('Nombre de clusters k')
ax2.set_ylabel('Silhouette')
ax2.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('clustering_elbow_silhouette.png', bbox_inches='tight')
plt.show()

k_optimal = list(K_range)[silhouettes.index(max(silhouettes))]
print(f'\n✅ k optimal (meilleure silhouette) : k = {k_optimal}')

### 1.2 Application de K-Means avec k optimal

In [ ]:
# On peut aussi choisir manuellement k ici si l'elbow est plus clair
K_FINAL = k_optimal  # Changer si besoin, ex: K_FINAL = 5

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(matrix_scaled)

# Ajouter les clusters au dataframe utilisateurs
user_clusters = pd.DataFrame({
    'userId'  : user_movie_matrix.index,
    'cluster' : cluster_labels
})

print(f'K-Means appliqué avec k={K_FINAL}')
print(f'Inertie finale        : {kmeans_final.inertia_:,.0f}')
print(f'Score de silhouette   : {silhouette_score(matrix_scaled, cluster_labels, sample_size=300, random_state=42):.4f}')
print('\nDistribution des clusters :')
print(user_clusters['cluster'].value_counts().sort_index())

### 1.3 Visualisation des clusters (PCA 2D)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(matrix_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels, cmap='tab10', s=30, alpha=0.7
)
plt.colorbar(scatter, label='Cluster')
plt.title(f'Visualisation des {K_FINAL} clusters (PCA 2D)')
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
plt.tight_layout()
plt.savefig('clustering_pca2d.png', bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée : clustering_pca2d.png')

### 1.4 Logique de recommandation par cluster

In [ ]:
# Fusion des clusters dans le dataframe principal
df_clustered = df.merge(user_clusters, on='userId')

def recommander_par_cluster(user_id, n=8):
    """
    Recommande les n films les mieux notés dans le même cluster
    que l'utilisateur cible, parmi ceux qu'il n'a pas encore vus.
    """
    # Cluster de l'utilisateur
    cluster = user_clusters.loc[user_clusters['userId'] == user_id, 'cluster'].values
    if len(cluster) == 0:
        return f'Utilisateur {user_id} introuvable.'
    cluster = cluster[0]

    # Films déjà vus par l'utilisateur
    films_vus = set(df_clustered.loc[df_clustered['userId'] == user_id, 'movieId'])

    # Notes moyennes des films dans le même cluster (hors utilisateur cible)
    meme_cluster = df_clustered[
        (df_clustered['cluster'] == cluster) &
        (df_clustered['userId'] != user_id)
    ]
    moyennes = (
        meme_cluster
        .groupby(['movieId', 'title'])['rating']
        .agg(note_moy='mean', nb_votes='count')
        .reset_index()
    )

    # Filtrer les films non vus + au moins 5 votes (fiabilité)
    recommandations = (
        moyennes[
            (~moyennes['movieId'].isin(films_vus)) &
            (moyennes['nb_votes'] >= 5)
        ]
        .sort_values('note_moy', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    recommandations.index += 1
    return cluster, recommandations[['title', 'note_moy', 'nb_votes']]


# === Exemple pour l'utilisateur 1 ===
USER_CIBLE = 1
cluster_user, recos = recommander_par_cluster(USER_CIBLE, n=8)

print(f'🎬 Recommandations pour l\'utilisateur {USER_CIBLE} (cluster {cluster_user})')
print(recos.to_string())

In [ ]:
# Visualisation des recommandations
fig, ax = plt.subplots(figsize=(10, 5))
recos_plot = recos.copy()
recos_plot['titre_court'] = recos_plot['title'].str[:40]
bars = ax.barh(recos_plot['titre_court'][::-1], recos_plot['note_moy'][::-1],
               color='steelblue', edgecolor='white')
ax.set_xlim(0, 5.5)
ax.set_xlabel('Note moyenne dans le cluster')
ax.set_title(f'Top recommandations — Utilisateur {USER_CIBLE} (cluster {cluster_user})')
for bar, val in zip(bars, recos_plot['note_moy'][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('recommandations_cluster.png', bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : recommandations_cluster.png')

---
# 🟢 PHASE 2 — Fouille de graphes
### 2.1 Construction du graphe biparti (utilisateurs ↔ films)

In [ ]:
# On filtre les notes >= 3.5 comme demandé dans le sujet
df_graph = df[df['rating'] >= 3.5].copy()
print(f'Arêtes (notes >= 3.5) : {len(df_graph):,} / {len(df):,} total')

# Pour Colab : on travaille sur un sous-ensemble pour la performance
# Prendre les 200 utilisateurs les plus actifs
top_users = df_graph.groupby('userId').size().nlargest(200).index
df_graph_sub = df_graph[df_graph['userId'].isin(top_users)]
print(f'Sous-graphe : {df_graph_sub["userId"].nunique()} utilisateurs, '
      f'{df_graph_sub["movieId"].nunique()} films, '
      f'{len(df_graph_sub):,} arêtes')

# Construction du graphe biparti
G = nx.Graph()

# Nœuds utilisateurs
for uid in df_graph_sub['userId'].unique():
    G.add_node(f'u_{uid}', bipartite=0, type='user')

# Nœuds films
for mid in df_graph_sub['movieId'].unique():
    titre = movies.loc[movies['movieId'] == mid, 'title'].values
    titre = titre[0] if len(titre) > 0 else str(mid)
    G.add_node(f'm_{mid}', bipartite=1, type='movie', title=titre)

# Arêtes (utilisateur, film) avec poids = note
for _, row in df_graph_sub.iterrows():
    G.add_edge(f'u_{int(row.userId)}', f'm_{int(row.movieId)}', weight=row.rating)

print(f'\n📊 Graphe construit :')
print(f'   Nœuds : {G.number_of_nodes():,} ({sum(1 for n,d in G.nodes(data=True) if d["type"]=="user")} users + {sum(1 for n,d in G.nodes(data=True) if d["type"]=="movie")} films)')
print(f'   Arêtes : {G.number_of_edges():,}')
print(f'   Densité : {nx.density(G):.5f}')

### 2.2 Mesures de centralité — Identification des films "hubs"

In [ ]:
print('⏳ Calcul des centralités...')

# Degree centrality
degree_cent = nx.degree_centrality(G)

# Betweenness centrality (sur un échantillon pour la performance)
betweenness_cent = nx.betweenness_centrality(G, k=100, normalized=True, seed=42)

# Filtrer uniquement les films
film_nodes = [n for n, d in G.nodes(data=True) if d['type'] == 'movie']

centralite_df = pd.DataFrame({
    'node'        : film_nodes,
    'title'       : [G.nodes[n].get('title', n) for n in film_nodes],
    'degree'      : [degree_cent[n] for n in film_nodes],
    'betweenness' : [betweenness_cent[n] for n in film_nodes]
}).sort_values('degree', ascending=False)

print('\n🏆 Top 10 films par degree centrality (films les plus connectés) :')
print(centralite_df[['title','degree','betweenness']].head(10).to_string(index=False))

In [ ]:
# Visualisation top 15 films par centralité
top15 = centralite_df.head(15).copy()
top15['titre_court'] = top15['title'].str[:35]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15['titre_court'][::-1], top15['degree'][::-1], color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Degree centrality')
ax.set_title('Top 15 films — Degree centrality (hubs du graphe)')
plt.tight_layout()
plt.savefig('graphe_centralite.png', bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : graphe_centralite.png')

### 2.3 PageRank — Classement des films par importance dans le réseau

In [ ]:
print('⏳ Calcul du PageRank...')
pagerank = nx.pagerank(G, alpha=0.85, weight='weight', max_iter=200)

pagerank_df = pd.DataFrame({
    'node'     : film_nodes,
    'title'    : [G.nodes[n].get('title', n) for n in film_nodes],
    'pagerank' : [pagerank[n] for n in film_nodes]
}).sort_values('pagerank', ascending=False)

print('\n📊 Top 10 films par PageRank :')
print(pagerank_df[['title','pagerank']].head(10).to_string(index=False))

### 2.4 Détection de communautés — Algorithme de Louvain

In [ ]:
print('⏳ Détection de communautés (Louvain)...')

partition = community_louvain.best_partition(G, weight='weight', random_state=42)

nb_communautes = len(set(partition.values()))
print(f'✅ Nombre de communautés détectées : {nb_communautes}')
print(f'   Modularité : {community_louvain.modularity(partition, G):.4f}')

# Distribution des tailles de communautés
from collections import Counter
tailles = Counter(partition.values())
print('\nTaille des communautés (top 10) :')
for com, taille in sorted(tailles.items(), key=lambda x: -x[1])[:10]:
    print(f'  Communauté {com:2d} : {taille} nœuds')

### 2.5 Recommandation par graphe

In [ ]:
def recommander_par_graphe(user_id, n=8):
    """
    Pour un utilisateur cible :
    1. Identifier sa communauté dans le graphe
    2. Recommander les films à forte centralité dans cette communauté
       que l'utilisateur n'a pas encore notés.
    """
    node_user = f'u_{user_id}'
    if node_user not in G.nodes:
        return f'Utilisateur {user_id} absent du graphe (pas dans le sous-ensemble).'

    # Communauté de l'utilisateur
    com_user = partition[node_user]

    # Tous les nœuds de cette communauté
    noeuds_com = [n for n, c in partition.items() if c == com_user]

    # Films déjà notés par l'utilisateur
    films_vus = set(G.neighbors(node_user))

    # Films de la communauté non encore vus, triés par PageRank
    candidats = [
        n for n in noeuds_com
        if G.nodes[n]['type'] == 'movie' and n not in films_vus
    ]

    candidats_df = pd.DataFrame({
        'node'     : candidats,
        'title'    : [G.nodes[c].get('title', c) for c in candidats],
        'pagerank' : [pagerank.get(c, 0) for c in candidats],
        'degree'   : [degree_cent.get(c, 0) for c in candidats]
    }).sort_values('pagerank', ascending=False).head(n).reset_index(drop=True)
    candidats_df.index += 1

    return com_user, candidats_df[['title', 'pagerank', 'degree']]


# === Exemple pour l'utilisateur 1 ===
USER_CIBLE_G = 1  # Changer si l'user n'est pas dans le sous-ensemble

# Trouver un utilisateur valide si besoin
users_dans_graphe = [int(n.replace('u_','')) for n in G.nodes if n.startswith('u_')]
if USER_CIBLE_G not in users_dans_graphe:
    USER_CIBLE_G = users_dans_graphe[0]
    print(f'ℹ️  User 1 absent du sous-graphe, on utilise user {USER_CIBLE_G}')

com, recos_graphe = recommander_par_graphe(USER_CIBLE_G, n=8)
print(f'🎬 Recommandations graphe pour utilisateur {USER_CIBLE_G} (communauté {com})')
print(recos_graphe.to_string())

### 2.6 Visualisation du graphe (sous-échantillon)

In [ ]:
# Visualiser un petit sous-graphe autour des films les plus centraux
top5_films = centralite_df['node'].head(5).tolist()

# Ego-graph : films centraux + leurs voisins directs
sous_g_nodes = set(top5_films)
for film in top5_films:
    voisins = list(G.neighbors(film))[:15]  # max 15 users par film
    sous_g_nodes.update(voisins)

SG = G.subgraph(sous_g_nodes)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(SG, seed=42, k=0.5)

# Couleurs : films = coral, users = bleu
colors = ['#E8593C' if SG.nodes[n]['type']=='movie' else '#4A90D9' for n in SG.nodes]
sizes  = [300 if SG.nodes[n]['type']=='movie' else 80 for n in SG.nodes]

nx.draw_networkx_nodes(SG, pos, node_color=colors, node_size=sizes, alpha=0.85)
nx.draw_networkx_edges(SG, pos, alpha=0.3, width=0.7, edge_color='gray')

# Labels uniquement pour les films centraux
labels = {n: G.nodes[n].get('title','')[:20] for n in top5_films if n in SG.nodes}
nx.draw_networkx_labels(SG, pos, labels, font_size=7, font_color='black')

# Légende
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E8593C', label='Film (hub)'),
    Patch(facecolor='#4A90D9', label='Utilisateur')
]
plt.legend(handles=legend_elements, loc='upper right')
plt.title('Sous-graphe biparti — Films centraux et leurs utilisateurs')
plt.axis('off')
plt.tight_layout()
plt.savefig('graphe_visualisation.png', bbox_inches='tight', dpi=130)
plt.show()
print('✅ Sauvegardé : graphe_visualisation.png')

---
# ⚖️ PHASE 3 — Comparaison des deux approches
*(Cette section sert à générer les données pour le rapport)*

In [ ]:
print('=== COMPARAISON DES RECOMMANDATIONS ===')
print()
print(f'Utilisateur : {USER_CIBLE_G}')
print()

# Approche 1 — Clustering
try:
    _, r_cluster = recommander_par_cluster(USER_CIBLE_G, n=8)
    titres_cluster = set(r_cluster['title'].tolist())
    print('📌 Recommandations par CLUSTERING :')
    for i, t in enumerate(r_cluster['title'], 1):
        print(f'  {i}. {t}')
except Exception as e:
    titres_cluster = set()
    print(f'  (Erreur clustering : {e})')

print()

# Approche 2 — Graphe
try:
    _, r_graphe = recommander_par_graphe(USER_CIBLE_G, n=8)
    titres_graphe = set(r_graphe['title'].tolist())
    print('📌 Recommandations par GRAPHE :')
    for i, t in enumerate(r_graphe['title'], 1):
        print(f'  {i}. {t}')
except Exception as e:
    titres_graphe = set()
    print(f'  (Erreur graphe : {e})')

print()
overlap = titres_cluster & titres_graphe
print(f'Overlap (films recommandés par les deux approches) : {len(overlap)}')
if overlap:
    for t in overlap:
        print(f'  • {t}')
else:
    print('  → Les approches sont COMPLÉMENTAIRES (peu de chevauchement)')

---
# 💾 Export des résultats
*(Pour le rapport et GitHub)*

In [ ]:
import os
os.makedirs('results', exist_ok=True)

# Clusters utilisateurs
user_clusters.to_csv('results/user_clusters.csv', index=False)

# Centralités films
centralite_df.to_csv('results/film_centralite.csv', index=False)

# PageRank films
pagerank_df.to_csv('results/film_pagerank.csv', index=False)

# Communautés
comm_df = pd.DataFrame(list(partition.items()), columns=['node', 'communaute'])
comm_df.to_csv('results/communautes.csv', index=False)

print('✅ Résultats exportés dans results/')
print('   - user_clusters.csv')
print('   - film_centralite.csv')
print('   - film_pagerank.csv')
print('   - communautes.csv')
print()
print('📊 Images sauvegardées :')
print('   - eda_overview.png')
print('   - clustering_elbow_silhouette.png')
print('   - clustering_pca2d.png')
print('   - recommandations_cluster.png')
print('   - graphe_centralite.png')
print('   - graphe_visualisation.png')
print()
print('🎉 Notebook terminé — Toutes les phases sont complètes !')